In [1]:
import sys
import numpy as np

sys.path.append('../../../src/')
from Rain.Rain import Rain
sys.path.pop()

from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

c:\Users\Menna\AppData\Local\Programs\Python\Python38\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "lazy",
      "params": {
        "num_of_workers": 1,
        "ips": ['40.87.71.244'],
        "ports": [50151]
        
      }
    },
  "temp_data_path": "../../../",
  "partitions": 1,
  "iterations": 2,
  "chunk_size": 1024*1024,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 20,
    "batch_size": 16,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/breast_cancer/train_data.npy"), np.load(
        "../../../data/breast_cancer/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/breast_cancer/test_data.npy"), np.load(
        "../../../data/breast_cancer/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 30
    num_labels = 2
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()
X_train = np.reshape(X_train, [-1, 30])
y_train = to_categorical(y_train)

In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-06 21:30:37,003 [DEBUG] [Rain] Rain is initialized
2023-07-06 21:30:37,005 [DEBUG] [Provisioner] Creating coordinator
2023-07-06 21:30:37,006 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData\coord/
2023-07-06 21:30:37,007 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-06 21:30:37,009 [DEBUG] [LazyProvisioner] Provisioner is initialized
2023-07-06 21:30:37,010 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData\divider/
2023-07-06 21:30:37,012 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData\divider/
2023-07-06 21:30:37,013 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData\divider/


In [8]:
# model = rain.train(X_train, y_train, strategy='async')

In [9]:
# X_test, y_test = get_test_data()
# X_test = np.reshape(X_test, [-1, 30])
# y_test = to_categorical(y_test)
# loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
# print("\nTest accuracy: %.1f%%" % (100.0 * acc))

In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-06 21:30:37,253 [INFO] [Provisioner] provisioner is serving
2023-07-06 21:30:37,253 [DEBUG] [Provisioner] Starting coordinator
2023-07-06 21:30:37,255 [INFO] [Coordinator] coordinator is serving
2023-07-06 21:30:37,256 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-06 21:30:37,264 [DEBUG] [Provisioner] Received 'NumOfWorkers: 1
' from the coordinator to define the number of workers
2023-07-06 21:30:37,267 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-06 21:30:37,269 [DEBUG] [LazyProvisioner] Creating 1 workers
2023-07-06 21:30:37,270 [DEBUG] [Provisioner] [Created workers]
IPs : ['40.87.71.244'], ports: [50151], statuses: [1], IDs : [1]
2023-07-06 21:30:37,284 [DEBUG] [DividerAmbassador] divider ambassador is serving
2023-07-06 21:30:37,286 [DEBUG] [DividerProxy] Training Started
2023-07-06 21:30:37,290 [DEBUG] [Coordinator] coordinator is sending workers info to divider
2023-07-06 21:30:37,293 [DEBUG]

In [11]:
X_test, y_test = get_test_data()
X_test = np.reshape(X_test, [-1, 30])
y_test = to_categorical(y_test)
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

RuntimeError: You must compile your model before training/testing. Use `model.compile(optimizer, loss)`.